# Mat & Comp – ML 5: Árboles de decisión
## Aprender a hacer las preguntas correctas

**Curso:** Machine learning y redes neuronales (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 4

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Leer un **árbol de decisión** y usarlo para clasificar, como un juego de preguntas de sí o no.
2. Medir qué tan «revuelto» está un grupo con el **índice de Gini**.
3. Programar desde cero la búsqueda de la **mejor pregunta** para dividir los datos.
4. Entrenar y dibujar árboles con **scikit-learn** e interpretar sus reglas.
5. Reconocer el **sobreajuste**: un árbol muy profundo memoriza los datos en lugar de aprender.

## 🔍 El juego de las preguntas

¿Has jugado a adivinar un personaje haciendo preguntas de **sí o no**? «¿Es una persona real?», «¿Es colombiano?», «¿Es deportista?»… Los buenos jugadores saben que **el orden importa**: una buena primera pregunta descarta a la mitad de los candidatos; una mala («¿Se llama Rigoberto?») casi no descarta a nadie.

En ML 4 clasificamos a los estudiantes preguntándoles a sus vecinos. El modelo acertaba bastante, pero no podía **explicar** su decisión. Hoy queremos un modelo que dé **reglas claras**, como:

> *Si estudias más de 6.75 horas por semana → apruebas.*

✏️ *Piensa en los 57 estudiantes de ML 1. Si solo pudieras hacer **una** pregunta de sí o no sobre sus hábitos para adivinar si aprobaron, ¿cuál harías? ¿Con qué número?*

**Mi pregunta:**

Un **árbol de decisión** es exactamente eso: una cadena de preguntas de sí o no, elegidas por la computadora para separar lo mejor posible los datos.

## 1. Los datos y una primera regla a mano

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
datos = pd.read_csv(url).dropna()
columnas = ["horas_estudio", "horas_sueno", "horas_redes"]
X = datos[columnas].to_numpy()
y = np.where(datos["nota_matematicas"] >= 3.0, "aprobó", "no aprobó")

print(len(datos), "estudiantes")
print(pd.Series(y).value_counts())

Una regla es simplemente un `if`. Probemos «si estudia más de 5 horas, aprobó» y contemos los aciertos con numpy:

In [ ]:
def regla(horas_estudio):
    if horas_estudio > 5:
        return "aprobó"
    else:
        return "no aprobó"

predicciones = np.array([regla(h) for h in X[:, 0]])
print(f"Aciertos: {np.mean(predicciones == y):.0%}")

¿Y si el número no es 5? Probemos **todos** los umbrales posibles y grafiquemos la precisión:

In [ ]:
umbrales = np.arange(0, 14.25, 0.25)
precisiones = [np.mean(np.where(X[:, 0] > u, "aprobó", "no aprobó") == y) for u in umbrales]

mejor = umbrales[np.argmax(precisiones)]
print(f"Mejor umbral: {mejor} horas → {max(precisiones):.1%} de aciertos")

plt.plot(umbrales, precisiones)
plt.axvline(mejor, color="red", linestyle="--")
plt.xlabel("umbral de horas de estudio")
plt.ylabel("precisión")
plt.title("«Aprobó si estudia más de ___ horas»")
plt.grid(alpha=0.3)
plt.show()

Con **una sola pregunta** acertamos el 91 %. Esa pregunta, por sí sola, ya es un árbol de decisión (uno muy pequeño, de profundidad 1).

Pero para hacer **más** preguntas necesitamos una forma de medir qué tan buena es cada división, no solo para la precisión final sino para los grupos que quedan. Para eso los árboles usan una medida del **desorden**.

## 2. Medir el desorden: el índice de Gini

Imagina una bolsa de canicas. Si todas son del mismo color, la bolsa está **pura**. Si hay mitad y mitad, está **revuelta**.

El **índice de Gini** responde: *si saco dos canicas al azar (devolviendo la primera), ¿cuál es la probabilidad de que sean de colores distintos?*

$$\text{Gini} = 1 - \sum_{\text{clases}} p^2$$

donde $p$ es la proporción de cada clase en el grupo.

- Todas iguales: $1 - 1^2 = 0$ (puro).
- Mitad y mitad: $1 - (0.5^2 + 0.5^2) = 0.5$ (lo más revuelto posible con dos clases).

In [ ]:
def gini(etiquetas):
    if len(etiquetas) == 0:
        return 0.0
    _, cuentas = np.unique(etiquetas, return_counts=True)
    p = cuentas / len(etiquetas)
    return 1 - np.sum(p ** 2)

print("Puro:          ", gini(np.array(["a", "a", "a", "a"])))
print("Mitad y mitad: ", gini(np.array(["a", "a", "b", "b"])))
print("3 contra 1:    ", gini(np.array(["a", "a", "a", "b"])))
print("Los 57 estudiantes:", round(gini(y), 3))

Al dividir un grupo en dos, el desorden total es el **promedio ponderado** del Gini de cada parte (un grupo grande pesa más que uno pequeño):

$$\text{Gini de la división} = \frac{n_{izq}}{n} \, \text{Gini}_{izq} + \frac{n_{der}}{n} \, \text{Gini}_{der}$$

La **mejor pregunta** es la que deja el Gini de la división **más bajo**.

In [ ]:
def gini_division(x, y, umbral):
    izq = y[x <= umbral]          # los que responden «sí» a «¿x ≤ umbral?»
    der = y[x > umbral]
    return (len(izq) * gini(izq) + len(der) * gini(der)) / len(y)

for u in [3, 5, 6.75, 9]:
    print(f"¿horas_estudio ≤ {u}?  →  Gini de la división = {gini_division(X[:, 0], y, u):.3f}")

## 3. La mejor pregunta, buscada por la computadora

Ahora probamos **todas** las preguntas posibles: cada variable y cada umbral. Como umbrales usamos los puntos medios entre valores consecutivos de los datos (no tiene sentido probar dos umbrales entre los mismos dos estudiantes).

In [ ]:
def mejor_division(X, y, nombres):
    mejor = (None, None, gini(y))            # (variable, umbral, Gini); empezamos sin dividir
    for j in range(X.shape[1]):
        valores = np.unique(X[:, j])          # valores distintos, ordenados
        for u in (valores[:-1] + valores[1:]) / 2:
            g = gini_division(X[:, j], y, u)
            if g < mejor[2]:
                mejor = (nombres[j], u, g)
    return mejor

variable, umbral, g = mejor_division(X, y, columnas)
print(f"Mejor pregunta: ¿{variable} ≤ {umbral}?   Gini: {gini(y):.3f} → {g:.3f}")

¡La computadora encontró la misma pregunta que nosotros! (Como los datos vienen de media en media hora, «≤ 6.75» separa a los mismos estudiantes que nuestro «> 6.5» de la sección 1: nadie estudia entre 6.5 y 7 horas.) Veamos cómo cambia el Gini de la división con cada variable:

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for j, eje in enumerate(ejes):
    valores = np.unique(X[:, j])
    us = (valores[:-1] + valores[1:]) / 2
    eje.plot(us, [gini_division(X[:, j], y, u) for u in us], "o-")
    eje.set_title(columnas[j])
    eje.set_xlabel("umbral")
    eje.grid(alpha=0.3)
ejes[0].set_ylabel("Gini de la división")
plt.show()

Las horas de estudio logran bajar mucho más el desorden que el sueño o las redes. Por eso la primera pregunta es sobre el estudio.

## 4. Seguir preguntando: el árbol crece

Después de la primera pregunta quedan dos grupos. Para cada uno, buscamos **otra vez** la mejor pregunta, con los mismos pasos. Así el árbol crece rama por rama.

In [ ]:
izq = X[:, 0] <= 6.75          # máscara: los que estudian 6.75 horas o menos
der = ~izq                     # ~ invierte la máscara: los demás

for nombre, grupo in [("Estudian ≤ 6.75 h", izq), ("Estudian > 6.75 h", der)]:
    etiquetas, cuentas = np.unique(y[grupo], return_counts=True)
    print(nombre, dict(zip(etiquetas.tolist(), cuentas.tolist())), f"Gini = {gini(y[grupo]):.3f}")
    v, u, g = mejor_division(X[grupo], y[grupo], columnas)
    print(f"   → siguiente pregunta: ¿{v} ≤ {u}?   (Gini {g:.3f})")

Nuestro árbol de profundidad 2 queda así:

```
¿horas_estudio ≤ 6.75?
├── sí (24 estudiantes: 3 aprueban, 21 no)
│     ¿horas_sueno ≤ 8.75?
│     ├── sí → NO APROBÓ
│     └── no → APROBÓ        (¡solo 3 estudiantes llegan aquí!)
└── no (33 estudiantes: 31 aprueban, 2 no)
      ¿horas_estudio ≤ 8.25?
      ├── sí → APROBÓ
      └── no → APROBÓ
```

Dos detalles curiosos:
- La rama de la derecha hace una pregunta cuyas dos respuestas dicen «aprobó». ¿Para qué? Porque deja un grupo **más puro** (los que estudian más de 8.25 horas aprueban todos), aunque no cambie la predicción.
- La regla «si estudias poco pero duermes más de 8.75 horas, apruebas» se basa en **solo 3 estudiantes**. ¿Te parece una regla confiable?

✏️ *Escribe el árbol como una función con `if` y `else` anidados.*

In [ ]:
def mi_arbol(estudio, sueno, redes):
    # Completa usando el árbol de arriba
    pass

## 5. Con scikit-learn

`DecisionTreeClassifier` hace todo esto automáticamente. Se usa igual que `KNeighborsClassifier`: crear, `fit`, `predict`. El parámetro `max_depth` limita cuántas preguntas seguidas puede hacer.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

arbol = DecisionTreeClassifier(max_depth=2, random_state=0)
arbol.fit(X, y)

print(export_text(arbol, feature_names=columnas))

In [ ]:
plt.figure(figsize=(12, 6))
plot_tree(arbol, feature_names=columnas, class_names=arbol.classes_, filled=True, rounded=True)
plt.show()

En cada caja del dibujo:
- la primera línea es la **pregunta** (si la respuesta es sí, se va a la izquierda);
- `gini` es el desorden de ese grupo;
- `samples` es cuántos estudiantes llegaron ahí;
- `value` cuenta cuántos hay de cada clase, en el orden `['aprobó', 'no aprobó']`;
- `class` es la predicción (la mayoría). El color indica la clase y es más intenso cuanto más puro es el grupo.

¡Son las mismas preguntas y los mismos umbrales que encontramos a mano!

In [ ]:
nuevo = [[8, 6, 3]]       # 8 h de estudio, 6 h de sueño, 3 h de redes
print("Predicción:", arbol.predict(nuevo))

## 6. ¿Cómo ve el plano un árbol?

Usemos solo dos variables (estudio y sueño) para poder dibujar el mapa de decisión, como en ML 4. Como cada pregunta compara **una** variable con un número, las fronteras son siempre líneas **horizontales o verticales**: el árbol corta el plano en rectángulos.

In [ ]:
X2 = X[:, :2]                                   # solo estudio y sueño
a, s = np.meshgrid(np.linspace(0, 14.5, 300), np.linspace(4.5, 10, 300))
rejilla = np.column_stack([a.ravel(), s.ravel()])

fig, ejes = plt.subplots(1, 3, figsize=(16, 5))
for eje, prof in zip(ejes, [1, 2, None]):
    modelo = DecisionTreeClassifier(max_depth=prof, random_state=0).fit(X2, y)
    Z = (modelo.predict(rejilla) == "aprobó").reshape(a.shape)
    eje.contourf(a, s, Z, levels=[-0.5, 0.5, 1.5], colors=["#f8d0d0", "#cdebd6"])
    for etiqueta, color in [("aprobó", "seagreen"), ("no aprobó", "crimson")]:
        sel = y == etiqueta
        eje.scatter(X2[sel, 0], X2[sel, 1], color=color, s=25, label=etiqueta)
    eje.set_title(f"max_depth = {prof}  ({modelo.get_n_leaves()} hojas)")
    eje.set_xlabel("horas de estudio")
ejes[0].set_ylabel("horas de sueño")
ejes[0].legend(loc="lower right")
plt.show()

`ravel()` aplana la cuadrícula en una lista larga de puntos para predecirlos todos de una vez, y `reshape` la vuelve a armar.

Con profundidad ilimitada (`None`), el árbol sigue preguntando hasta que **cada** grupo es puro: dibuja rectángulos diminutos alrededor de estudiantes sueltos. ¿Está aprendiendo o memorizando?

## 7. ⚠️ Sobreajuste: el árbol que memoriza

Midamos la precisión de dos formas para distintas profundidades:
- **con los mismos datos** con que se entrenó (`arbol.score(X, y)`);
- **dejando uno afuera**, como en ML 4: entrenamos sin el estudiante i y vemos si acertamos con él.

In [ ]:
def precision_dejando_uno(prof):
    aciertos = 0
    for i in range(len(X)):
        X_otros = np.delete(X, i, axis=0)
        y_otros = np.delete(y, i)
        modelo = DecisionTreeClassifier(max_depth=prof, random_state=0).fit(X_otros, y_otros)
        if modelo.predict(X[i:i+1])[0] == y[i]:
            aciertos += 1
    return aciertos / len(X)

profundidades = [1, 2, 3, 4, 5, 6]
en_entrenamiento = []
dejando_uno = []
for p in profundidades:
    modelo = DecisionTreeClassifier(max_depth=p, random_state=0).fit(X, y)
    en_entrenamiento.append(modelo.score(X, y))
    dejando_uno.append(precision_dejando_uno(p))
    print(f"profundidad {p}: con los mismos datos {en_entrenamiento[-1]:.0%}   ·   dejando uno afuera {dejando_uno[-1]:.0%}")

plt.plot(profundidades, en_entrenamiento, "o-", label="con los mismos datos")
plt.plot(profundidades, dejando_uno, "s-", label="dejando uno afuera")
plt.xlabel("profundidad máxima del árbol")
plt.ylabel("precisión")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Más preguntas no siempre es mejor")
plt.show()

Esta gráfica es una de las más importantes del machine learning:
- Con los mismos datos, el árbol profundo llega al **100 %**: memorizó a cada estudiante.
- Con estudiantes que **no vio**, empeora: las reglas basadas en uno o dos estudiantes no sirven para los demás.

Eso es el **sobreajuste**: aprender de memoria los detalles (y el ruido) de los datos en lugar de la regla general. Es como estudiar para un examen memorizando las respuestas de un taller: sacas 5 en el taller y fracasas con preguntas nuevas. En ML 7 lo estudiaremos a fondo.

## 8. ¿Qué variables importan?

Los árboles tienen una ventaja extra: dicen **cuánto ayudó cada variable** a reducir el desorden. Esto se llama **importancia** de las variables y suma 1.

In [ ]:
arbol3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
for nombre, imp in zip(columnas, arbol3.feature_importances_):
    print(f"{nombre:15} {imp:.2f}  " + "█" * int(imp * 40))

Las horas de estudio explican casi todo, igual que vimos con las correlaciones en ML 1 (0.90 con la nota).

## 9. Errores comunes

**Error 1: usar texto como variable.** scikit-learn solo acepta números. La columna `deporte` dice `"sí"` o `"no"`:

In [ ]:
X_con_deporte = datos[["horas_estudio", "deporte"]].to_numpy()
DecisionTreeClassifier().fit(X_con_deporte, y)

La solución es convertirla en números: `(datos["deporte"] == "sí").astype(int)` da 1 para «sí» y 0 para «no».

**Error 2: creerle al 100 %.** Una precisión perfecta con los mismos datos de entrenamiento casi siempre es sobreajuste, no un modelo perfecto.

**Error 3: confiar en hojas con muy pocos datos.** Una regla que se basa en 1, 2 o 3 casos es frágil. Se puede evitar con `max_depth` o con `min_samples_leaf=5` (cada hoja debe tener al menos 5 datos).

**Error 4: leer mal el dibujo.** En `plot_tree`, la izquierda es siempre la respuesta **sí** (se cumple la condición) y la derecha la respuesta **no**.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …calculas `gini` de un grupo con **tres** clases en partes iguales, como `["a", "b", "c"]`? ¿Es mayor o menor que 0.5?
2. …entrenas un árbol con `max_depth=1` y miras `export_text`? ¿Qué pregunta hace?
3. …usas `min_samples_leaf=5` en lugar de `max_depth`? ¿Sigue apareciendo la regla del sueño de 3 estudiantes?
4. …multiplicas las horas de sueño por 60 (minutos), como en ML 4? ¿Cambian las predicciones del árbol? ¿Por qué es distinto a k-NN?
5. …le preguntas al árbol de profundidad 2 por un estudiante con 20 horas de estudio, 3 de sueño y 10 de redes?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). Gini a mano.** Un grupo tiene 6 estudiantes que aprobaron y 2 que no. Calcula su índice de Gini con lápiz y compruébalo con la función `gini`. Luego calcula el Gini de un grupo con 1 y 7.

**Ejercicio 2 (fácil). Tu árbol contra el de la computadora.** Completa `mi_arbol` (sección 4), predice con ella a los 57 estudiantes y calcula su precisión. ¿Coincide con `arbol.score(X, y)`?

**Ejercicio 3 (medio). ¿Importa el deporte?** Agrega la columna `deporte` convertida en 0 y 1 a las tres variables. Entrena un árbol con `max_depth=3`, dibújalo y mira la importancia de cada variable. ¿El árbol usa el deporte? Compara con lo que vimos en ML 1 (los deportistas tienen un promedio un poco mayor).

**Ejercicio 4 (medio). Árbol contra vecinos.** Con las frutas de ML 4, entrena un árbol con `max_depth=3` y dibuja su mapa de decisión. Compáralo con el de k-NN: ¿qué forma tienen las fronteras? ¿Qué predice el árbol para la fruta nueva `[7.0, 7.1]`?

```python
rng = np.random.default_rng(4)
n = 20
mandarinas = np.column_stack([rng.normal(7.4, 0.55, n), rng.normal(6.6, 0.5, n)]).round(1)
limones    = np.column_stack([rng.normal(6.0, 0.5, n),  rng.normal(7.9, 0.55, n)]).round(1)
Xf = np.vstack([mandarinas, limones])
yf = np.array(["mandarina"] * n + ["limón"] * n)
```

**Ejercicio 5 (medio). Árboles que predicen números.** `DecisionTreeRegressor` (de `sklearn.tree`) predice la **nota** en lugar de aprobó o no: en cada hoja predice el **promedio** de las notas que llegaron ahí. Entrénalo con solo `horas_estudio` y `max_depth` = 1, 2 y 4, y grafica sus predicciones para `np.linspace(0, 14, 300)` junto con los puntos. ¿Qué forma tienen las curvas? ¿Cuál se parece más a la recta de ML 2?

## 🏆 Reto opcional: las flores iris

El conjunto de datos **iris** es el más famoso del machine learning: 150 flores de tres especies, con 4 medidas de sus pétalos y sépalos. Viene incluido en scikit-learn.

```python
from sklearn.datasets import load_iris
iris = load_iris()
Xi, yi = iris.data, iris.target_names[iris.target]
print(iris.feature_names)
```

1. Entrena un árbol con `max_depth=2` y dibújalo con `plot_tree`. ¿Qué medida usa en la primera pregunta? ¿Qué especie queda separada por completo con una sola pregunta?
2. Calcula la precisión dejando uno afuera para profundidades de 1 a 6. ¿Cuál elegirías?
3. Con tus propias funciones `gini` y `mejor_division`, busca la mejor primera pregunta. ¿Es la misma que eligió scikit-learn? Si no lo es, calcula el Gini de la división de las dos preguntas y explica qué pasó.

## 🎬 Cierre

Hoy aprendiste que:
- Un **árbol de decisión** clasifica con una cadena de preguntas de sí o no, y sus reglas se pueden **leer y explicar**.
- El **índice de Gini** mide el desorden de un grupo: 0 si es puro, 0.5 si está mitad y mitad.
- El árbol elige en cada paso la pregunta que **más reduce el desorden**, probando todas las variables y todos los umbrales.
- Las fronteras de un árbol son rectángulos, y no le afecta la escala de las variables.
- Un árbol demasiado profundo **sobreajusta**: memoriza los datos y falla con datos nuevos.

**Para pensar:** en ML 4 y en esta lección siempre teníamos la **respuesta correcta** de cada estudiante (aprobó o no) para aprender de ella. Eso se llama aprendizaje **supervisado**. ¿Pero qué pasa si nadie nos da las respuestas? ¿Puede la computadora descubrir sola que los datos forman **grupos**? En la próxima lección veremos el agrupamiento con **k-means**. 🔍